# Freshmart data analysis
We have been presented with a business problem where Freshmart stores want to understand if customer shopping baskets have declined or not. This analysis will help answer the hypotheses developed around this problem. There are three different sheets that will be cleaned separately and joined for further analysis. There is an increase in the number of people visiting the stores, but there is no growth in revenue.

# Importing libraries
It is important to import essential libraries that are useful for data processing, mathematical calculations and visualisation. 

In [0]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

# Transaction dataset

## Data ingestion
Transactions dataset is brought into python notebook from delta spark table and converted to pandas for further processing.

In [0]:
# importing freshmart transactions table and converting it to pandas
transactions = spark.table("freshmart_supermarket.data.transactions").toPandas()

## Data inspection/understanding
This section explores transactions dataset to have an understanding of what it contains. For example the first few rows, columns, data types, summaries, etc.

In [0]:
#checking the first few rows
transactions.head()

In [0]:
# checking columns in transaction dataset
transactions .columns

In [0]:
# Checking the size of the dataset
transactions.shape

In [0]:
#checking the data types...transaction date is recorderd as an object, transformation to date format will be needed. basket_value_zar is also recorded as an object, will need to be transformed to float
transactions.dtypes


In [0]:
#check for a summary of the dataset
transactions.info()

In [0]:
#calculating numerical summaries
transactions.describe()

In [0]:
# check the first day of data collection
transactions["transaction_date"].min()


In [0]:
# check the last day of data collection
transactions["transaction_date"].max()

In [0]:
#checking the distinct values in the is_loyalty_member column
transactions["is_loyalty_member"].value_counts()

In [0]:
#checking the distinct payment methods
transactions["payment_method"].value_counts()

## Data cleaning
Based on the findings in the previous section, we found that there are null values and none standardised date and basket values. These will be delt with in this section. Additionally, duplicates will be checked in this section.  

In [0]:
#checking for duplicates...there are no duplicate rows in the dataset
transactions.duplicated().sum()

In [0]:
#checking for null values...customer_id has a big number of null values almost 50% but we cannot replace it because every customer id is unique to the customer. The rest of the columns do not contain any null values
transactions.isnull().sum()

In [0]:
# transforming date column from object to date type and checking if the transformation has been effected
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions.dtypes

In [0]:
# transforming basket_value_zar from an object to a numeric float and checking if the change has been effected
transactions["basket_value_zar"] = pd.to_numeric(transactions["basket_value_zar"]).round(2)
transactions.dtypes

In [0]:
# confirmation of the effected transformations
transactions.head()

## Feature engineering
In this section, new features will be created such as months, month name, years, day of week, half yearly.

In [0]:
# extracting year from date
transactions["year"] = transactions["transaction_date"].dt.year
print(transactions["year"])

In [0]:
# extracting month
transactions["month"] = transactions["transaction_date"].dt.month
print(transactions["month"])

In [0]:
#extracting month_name
transactions["month_name"] = transactions["transaction_date"].dt.month_name()
print(transactions["month_name"])

In [0]:
# creating half year buckets 
transactions["half_year"] = transactions ["month"]. apply (lambda x: "H1" if x <= 6 else "H2")


In [0]:
# creating day of week buckets
transactions ["day_type"] = transactions["day_of_week"].apply(lambda x:"Weekend" if x in ["Saturday", "Sunday"] else "Weekday")

# Stores dataset

## Data ingestion
Stores dataset is brought into python notebook from delta spark table and converted to pandas for further processing.

In [0]:
# importing freshmart transactions table and converting it to pandas
stores = spark.table("freshmart_supermarket.data.stores").toPandas()

## Understanding stores dataset

In [0]:
#explore the first columns
stores.head()

In [0]:
# checking columns in stores dataset
stores .columns

In [0]:
# Checking the size of stores dataset
stores.shape

In [0]:
#checking the data types in stores dataset..competitor_open_date should be transformed to date
stores.dtypes

In [0]:
#checking the summary of the dataset
stores.info()

In [0]:
# calculating numerical summaries
stores.describe()

In [0]:
# checking unique stores
stores["store_name"].value_counts()

In [0]:
# checking the unique provinces
stores["province"].value_counts()

In [0]:
# checking the unique managers and their stores
stores[["region_manager", "province"]].value_counts()

In [0]:
# checking the unique store format
stores["store_format"].value_counts()

In [0]:
# checking the unique opened year
stores["opened_year"].value_counts()

In [0]:
# cheque the unique nearby competitor
stores["has_nearby_competitor"].value_counts()

## Data cleaning


In [0]:
# checking for duplicates....there are no duplicates in the dataset
stores.duplicated(). sum()

In [0]:
#checking for null values....nulls are found in the competitor open date column but it is because they are the ones were there was no competitor opening nearby so there is nothing that we are going to do about them.
stores.isnull(). sum()

In [0]:
# transforming date column from object to date type and checking if the transformation has been effected
stores["competitor_open_date"] = pd.to_datetime(stores["competitor_open_date"])
stores.dtypes

## Feature engineering

In [0]:
# extract competitor open year
stores["competitor_open_year"] = stores["competitor_open_date"].dt.year.astype("Int64")

In [0]:
# checking to see if the transformations were successfully effected
stores.head()

# Stockout Dataset

### Data Ingestion
Stockouts dataset is brought into python notebook from delta spark table and converted to pandas for further processing.

In [0]:
# improting dataset
stockouts=spark.table("freshmart_supermarket.data.stockout_events_fresh_mart_dataset_1").toPandas()

## Inspecting/Understanding data

In [0]:
# chcking the first few rows
stockouts.head()

In [0]:
# checking the columns in the stockout dataset
stockouts.columns

In [0]:
# checking the size of the dataset
stockouts.shape

In [0]:
# checking the data types...date is recorded as an object, it will need to be transformed to date
stockouts.dtypes

In [0]:
# checking dataset summary
stockouts.info()

In [0]:
# checking for duration summary
stockouts.describe()

In [0]:
# checking the unique stockouts categories
stockouts["category"]. unique()


In [0]:
# checking minimum stockout dates
stockouts["date"].min()

In [0]:
# checking the last stockout date
stockouts["date"].max()

In [0]:
# checking out the duration of stockouts
len(stockouts["date"])

In [0]:
# checking for duplicates
stockouts.duplicated().sum()

### Data inspection summary
* Date column is saved as an object, it will be converted to datetime
* The dataset does not contain any null values 
* There are no missing values in the dataset, no further cleaning will be done for missing values
* The dataset does not contain any duplicates, thus no cleaning of duplicates will be done.

# Data cleaning

* Date column is saved as an object, it will be transformed to date


In [0]:
# transforming date column to datetime
stockouts["date"] = pd.to_datetime(stockouts["date"])

In [0]:
# checking to ensure that the transformation was effected
stockouts.info()

### Stockouts data cleaning summary
* Date column was transformed to datetime

# Hypotheses development and testing

### Hypothesis 1. Shoppers now buy fewer items per visit

### Checking if there are monthly changes in shopping baskets

In [0]:
# checking if there are monthly changes in the shopping basket
monthly = transactions.groupby("month").agg(
    transactions=("transaction_id", "count"),
    revenue=("basket_value_zar", "sum"),
    average_basket=("basket_value_zar", "mean"),
    average_items=("num_items", "mean")
)

print(monthly.round(0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

monthly.reset_index().plot(x="month", y="average_basket", marker="o",
                            ax=axes[0], legend=False)
axes[0].set(title="Average basket value by month",
            xlabel="Month", ylabel="Rand per transaction")

monthly.reset_index().plot(x="month", y="average_items", marker="o",
                            ax=axes[1], legend=False)
axes[1].set(title="Average items per basket by month",
            xlabel="Month", ylabel="Items per transaction")

plt.tight_layout()
plt.show()

### Checking if there are half yearly changes in shopping baskets

In [0]:
# checking if there are changes in the shopping baskets between the two periods of the year
half_year_results = transactions.groupby("half_year").agg(
    transactions=("transaction_id", "count"),
    revenue=("basket_value_zar", "sum"),
    average_basket=("basket_value_zar", "mean"),
    average_items=("num_items", "mean")
)

print(half_year_results.round(0))

# Visualizing half-yearly comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

half_year_results.reset_index().plot(x="half_year", y="average_basket", kind="bar",
                                      ax=axes[0], legend=False, color=["blue", "orange"])
axes[0].set(title="Average basket value by half year",
            xlabel="Period", ylabel="Rand per transaction")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

half_year_results.reset_index().plot(x="half_year", y="average_items", kind="bar",
                                      ax=axes[1], legend=False, color=["blue", "orange"])
axes[1].set(title="Average items per basket by half year",
            xlabel="Period", ylabel="Items per transaction")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

 The results show that there has been a gradual decrease in both the basket value and average number of items. The average units bought droped from 15 items to 12. Results from half_yearly tests also shows that there has been an average reduction in the number of items in the shopping buskets from an average of 14 items to 13. This test confirms the hypotheses that shopers are now buying a few items per easch store visit.

# Joining transactions and stores tables

In [0]:
df = transactions.merge(stores, on="store_id", how="left")

print("Transactions before join:", len(transactions))
print("Rows after join:", len(df))

In [0]:
for column in [
    "province",
    "store_format",
    "is_loyalty_member",
    "day_type",
    "store_name"
]:
    print("Results by:", column)

    result = df.groupby(["half_year", column]).agg(
        transactions=("transaction_id", "count"),
        average_basket=("basket_value_zar", "mean"),
        average_items=("num_items", "mean")
    ).reset_index()

    display(result.round(2))

In [0]:
# visualizing average basket value and average items by category, H1 vs H2
for column in [
    "province",
    "store_format",
    "is_loyalty_member",
    "day_type",
    "store_name"
]:
    result = df.groupby(["half_year", column]).agg(
        average_basket=("basket_value_zar", "mean"),
        average_items=("num_items", "mean")
    ).reset_index()

    pivot_basket = result.pivot(index=column, columns="half_year", values="average_basket")
    pivot_items = result.pivot(index=column, columns="half_year", values="average_items")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pivot_basket.plot(kind="bar", ax=axes[0], color=["steelblue", "orange"])
    axes[0].set_title(f"Average Basket Value by {column.capitalize()}", fontweight="bold")
    axes[0].set_ylabel("Average Basket Value (ZAR)")
    axes[0].set_xlabel(column.capitalize())
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
    axes[0].legend(title="Half Year")
    axes[0].grid(axis="y", alpha=0.3)

    pivot_items.plot(kind="bar", ax=axes[1], color=["steelblue", "orange"])
    axes[1].set_title(f"Average Items per Basket by {column.capitalize()}", fontweight="bold")
    axes[1].set_ylabel("Average Items")
    axes[1].set_xlabel(column.capitalize())
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
    axes[1].legend(title="Half Year")
    axes[1].grid(axis="y", alpha=0.3)

    # add data labels for categories with few bars (skip store_name which has too many)
    if column != "store_name":
        for container in axes[0].containers:
            axes[0].bar_label(container, fmt="R%.0f", padding=3, fontsize=8)
        for container in axes[1].containers:
            axes[1].bar_label(container, fmt="%.1f", padding=3, fontsize=8)

    plt.tight_layout()
    plt.show()

### Hypothesis 2. Small buckets became more common

#### Checking whether small buckets have become common

In [0]:
# calculating the shopping buckets to see if there was an increase in small buckets and a comparison across all buckets. 
df["basket_type"] = pd.cut(
    df["basket_value_zar"],
    bins=[0, 100, 400, float("inf")],
    labels=["Small", "Medium", "Large"],
    include_lowest=True
)
basket_mix = pd.crosstab(
    df["half_year"],
    df["basket_type"],
    normalize="index"
) * 100

basket_mix = basket_mix.reindex(["H1", "H2"]).reset_index()

display(basket_mix.round(1))

# Visualizing basket type comparison
ax = basket_mix.set_index('half_year')[['Small', 'Medium', 'Large']].plot(kind='bar', figsize=(10, 6))
plt.title('Basket Type Distribution: H1 vs H2')
plt.xlabel('Period')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Basket Type')

# Add labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

Small baskets: increased from 23.1% in H1 to 28.2% in H2 (+5.1%)
Medium baskets: remained stable at around 45%
Large baskets: decreased from 31.7% in H1 to 26.7% in H2 (-5.0%)
This confirms the hypothesis that small trips have become more common - there's a clear 5-point increase in small basket transactions from the first half to the second half of the year!

### Hypothesis 3. Shopping baskets are affected by competitors opening

In [0]:
# checking store performance before nd after competitors
competitor_data = df[df["has_nearby_competitor"] == "Yes"].copy()

competitor_data["period"] = np.where(
    competitor_data["transaction_date"] < competitor_data["competitor_open_date"],
    "Before",
    "After"
)

result = competitor_data.groupby("period").agg(
    transactions=("transaction_id", "count"),
    average_basket=("basket_value_zar", "mean")
).reset_index()

display(result.round(2))

In [0]:
# checking out individually affected stores before and after competitor opening
store_results = competitor_data.groupby(
    ["store_id", "store_name", "period"]
)["basket_value_zar"].mean().reset_index()

display(store_results.round(2))

# Visualizing store performance before and after competitor opening
fig, ax = plt.subplots(figsize=(12, 6))

# Creating a unique identifier combining store_id and store_name
store_results['store_label'] = store_results['store_id'].astype(str) + ' - ' + store_results['store_name']

# Pivoting data for grouped bar chart
store_pivot = store_results.pivot(index='store_label', columns='period', values='basket_value_zar')

# Creating grouped bar chart
store_pivot.plot(kind='bar', ax=ax, color=['blue', 'orange'])

ax.set_title('Average Basket Value by Store: Before vs After Competitor Opening', fontsize=14, fontweight='bold')
ax.set_xlabel('Store', fontsize=12)
ax.set_ylabel('Average Basket Value (ZAR)', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Period')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

Overall, the average basket value dropped from R285.58 → R225.09 (21% decrease). Transaction volume decreased significantly (14,263 → 6,339). In addition all 12 stores showed decreased basket values after competitor opening. Declines ranged from R38 to R74 per basket with drops showing accross all affected stores. Given the findings on the overall and individual store performance, we confirm that basket values fell after nearby competitors opened. hence accepting the Hypotheses that shopping baskets are affected by competitors opening.

### Merging the new dataframe (transactions and stores) with stockouts

The created df is now mergd with stockouts to avoid duplicates

In [0]:
# merging df and stockouts
stockout_days = stockouts[["store_id", "date"]].drop_duplicates()

final_df = df.merge(
    stockout_days,
    left_on=["store_id", "transaction_date"],
    right_on=["store_id", "date"],
    how="left"
)

print("Rows after stockout join:", len(final_df))

### Hypothesis 4. Shopping baskets are affected by stockouts

In [0]:
final_df["stockout_day"] = np.where(
    final_df["date"].notna(),
    "Recorded stockout",
    "No recorded stockout"
)

stockout_results = final_df.groupby("stockout_day").agg(
    transactions=("transaction_id", "count"),
    average_basket=("basket_value_zar", "mean"),
    average_items=("num_items", "mean")
).reset_index()

display(stockout_results.round(2))

# Visualizing stockout impact
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

stockout_results.plot(x='stockout_day', y='average_basket', kind='bar', 
                       ax=axes[0], legend=False, color=['blue', 'orange'])
axes[0].set_title('Average Basket Value: Stockout vs No Stockout', fontsize=14)
axes[0].set_ylabel('Average Basket Value (ZAR)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

for container in axes[0].containers:
    axes[0].bar_label(container, fmt='R%.2f', padding=3)

stockout_results.plot(x='stockout_day', y='average_items', kind='bar',
                       ax=axes[1], legend=False, color=['blue', 'orange'])
axes[1].set_title('Average Items per Basket: Stockout vs No Stockout', fontsize=14)
axes[1].set_ylabel('Average Items per Basket')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.2f', padding=3)

plt.tight_layout()
plt.show()

Stockout days have lower average basket values. When customers encounter stockouts-they spend approximately R43 less per visit and they purchase about 2 fewer items per basket. We therefore accept the hypotheses that stockouts affects shopping baskets.